# 01 ・ 影城 API：向沒有文件的 API 要資料

## 這一章要做什麼

抓兩家影城「現在上映哪些電影」，產出一份乾淨的片名清單。

| 影城 | 端點 | 方法 |
|---|---|---|
| 秀泰 | `capi.showtimes.com.tw/4/app/bootstrap` | GET |
| 美麗華 | `www.miramarcinemas.tw/api/Booking/GetMovie/` | POST |

## 為什麼從這裡開始

這兩支是**沒有公開文件的內部 API**。沒有說明書、沒有保證、隨時可能改。
這其實才是真實世界抓資料的常態，而處理這種 API 的技巧
（觀察結構、遞迴搜尋、偽裝成瀏覽器、清理髒資料）
在後面每一章都會用到。

下一章的 TMDB 剛好相反 —— 有完整文件、有金鑰、有速率限制的正規 API。

## 本章產出

`movieapp/sources.py`，提供：

```python
sources.showtimes()        # -> (片名清單, 錯誤)
sources.miramar()          # -> (片名清單, 錯誤)
sources.titles_by_source() # -> ({來源: 片名清單}, 錯誤)
sources.all_titles()       # -> (合併去重後的清單, 錯誤)
```

---
## 0 ・ 開場

In [ ]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.insert(0, "..")
from movieapp.config import setup; setup()

---
## 1 ・ 先看一眼原始回應

寫任何解析程式之前，先看看對方到底回了什麼。這一格刻意用最原始的
`requests.get()`，先體會沒有工具時的樣子。

秀泰這支要帶 `User-Agent`。不帶的話會被認出不是瀏覽器而擋掉 ——
這在沒有文件的 API 上很常見，通常要試過才知道。

In [ ]:
import requests

SHOWTIMES_URL = "https://capi.showtimes.com.tw/4/app/bootstrap"
BROWSER_UA = ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
              "(KHTML, like Gecko) Chrome/126.0 Safari/537.36")

try:
    resp = requests.get(SHOWTIMES_URL,
                        headers={"User-Agent": BROWSER_UA, "Accept": "application/json"},
                        timeout=20)
    print("HTTP 狀態碼 :", resp.status_code)
    print("回應大小     :", f"{len(resp.content) / 1024:.0f} KB")
    raw = resp.json()
    print("最外層的 key :", list(raw))
except Exception as exc:
    raw = None
    print(f"連線失敗（{type(exc).__name__}）：{exc}")
    print("這一章全程需要連線。先確認網路，再重跑這一格。")

**約 1.8 MB、最外層只有兩個 key。** 電影在哪？完全看不出來。

把 `payload` 拆開看看裡面有什麼。

In [ ]:
import json
from movieapp.config import pad     # 中英混排的表格要用它才不會歪

if raw:
    payload = raw["payload"]
    print(pad("分支名稱", 26) + pad("型別", 8) + pad("筆數", 8) + pad("大小", 10, "right"))
    print("-" * 52)
    for key, value in payload.items():
        size = len(json.dumps(value, ensure_ascii=False))
        count = len(value) if isinstance(value, (list, dict)) else "-"
        shown = f"{size / 1024:.0f} KB" if size >= 1024 else f"{size} B"
        print(pad(key, 26) + pad(type(value).__name__, 8)
              + pad(count, 8) + pad(shown, 10, "right"))

兩件事值得注意：

1. `programs`（70 筆）看起來最像電影清單
2. `eventsForCorporations` 一支就佔了整個回應的 **91%**，卻跟電影無關

第 2 點很重要：**我們要的資料只佔回應的一小部分。**
對方多塞了 1.7 MB 跟電影無關的東西，而我們每次都得整包收下來 ——
沒有文件的 API 就是這樣，你不能挑，只能拿到之後自己找。

> `pad()` 是什麼？中文字在畫面上佔兩格，但 `len()` 只算一個字元，
> 所以用 `ljust()` 或 f-string 的 `:<26` 排中英混雜的表格一定會歪。
> `pad()` 依實際顯示寬度補空白，後面幾章印表格都會用到。

---
## 2 ・ 遞迴搜尋：不管藏多深都撈得到

現在知道電影在 `payload.programs[]` 裡。但要怎麼取？

**寫法 A：寫死路徑**
```python
titles = [p["name"] for p in data["payload"]["programs"]]
```
簡短，但對方只要把 `programs` 搬一層或改個包裝，整個就壞掉。

**寫法 B：遞迴找 key**
不管藏多深，只要 JSON 裡出現目標欄位就撈出來。

沒有文件的 API 用寫法 B 比較耐撞。代價是**會撈到你沒預期的東西**，
所以撈完一定要檢查 —— 等一下就會踩到這個坑。

先用一份小資料理解遞迴怎麼運作：

In [ ]:
def collect_keys(obj, target_key, result=None):
    """遞迴走訪整個 JSON，收集所有名為 target_key 的欄位值。"""
    if result is None:
        result = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == target_key:
                result.append(value)
            collect_keys(value, target_key, result)   # 值本身可能還有巢狀
    elif isinstance(obj, list):
        for item in obj:
            collect_keys(item, target_key, result)

    return result


# 用一份刻意做得很亂的小資料測試：name 分別在第 1、2、3、4 層
demo = {
    "name": "最外層",
    "box": {"name": "第二層", "items": [{"name": "清單裡"}, {"deep": {"name": "更深處"}}]},
    "noise": [1, 2, "字串不會被誤判", None],
}

print(collect_keys(demo, "name"))

四個不同深度的 `name` 都撈到了，而且 `noise` 裡的數字、字串、`None`
都不會讓遞迴出錯 —— 因為只有 `dict` 和 `list` 會繼續往下走。

---
## 3 ・ 第一個坑：撈到太多東西

`programs[]` 裡的電影有兩個片名欄位，先看看它們長什麼樣：

In [ ]:
if raw:
    program = raw["payload"]["programs"][0]
    print("一部電影有這些欄位：")
    print("  ", list(program))
    print()
    print(pad("name（中文）", 34) + "nameAlternative（英文）")
    print("-" * 76)
    for p in raw["payload"]["programs"][:8]:
        print(pad(str(p.get("name"))[:32], 34) + str(p.get("nameAlternative")))

中文名在 `name`。那直接遞迴撈 `name` 就好了吧？試試看：

In [ ]:
if raw:
    found = collect_keys(raw, "name")
    print(f"撈到 {len(found)} 筆 —— 但電影只有 70 部")
    print()
    print("看看撈到了什麼：")
    for value in found[:6]:
        print("   ", repr(value))
    print("    ...")
    for value in found[100:106]:
        print("   ", repr(value))

**撈到 224 筆。** 因為 `name` 是個到處都有的通用欄位 ——
影城名稱、常見問題、場館、電影類型全都叫 `name`。

這就是遞迴搜尋的代價。前一章的 `nameAlternative` 剛好夠特別所以沒事，
`name` 就不行了。

**解法：先縮小範圍，再撈。** 用結構特徵找出「哪一個陣列是電影清單」——
它是個 list、元素是 dict、而且帶有 `nameAlternative` 欄位。
用結構判斷而不是寫死路徑，對方搬動位置時還是找得到。

In [ ]:
def find_programs(data):
    """在回應裡找出電影清單。"""
    for candidate in collect_keys(data, "programs"):
        if (isinstance(candidate, list) and candidate
                and isinstance(candidate[0], dict)
                and "nameAlternative" in candidate[0]):
            return candidate
    return []


if raw:
    programs = find_programs(raw)
    names = [p.get("name") for p in programs]
    print(f"縮小範圍後：{len(names)} 筆")
    for name in names[:8]:
        print("   ", repr(name))

---
## 4 ・ 第二個坑：片名很髒

看看上面的結果，有沒有發現問題？

- `'驀然回首 友誼場'`、`'驀然回首 攜手投稿場'` —— 同一部片的不同場次
- `'藍色監獄 Ado應援特別場'` —— 活動名稱黏在片名後面
- `'劇場版 吉伊卡哇 人魚島的秘密 (國語版)'` —— 語言版本註記
- `'攻殼機動隊(1995) 4K 數位修復版'` —— 年份 + 畫質 + 版本，三層都要剝

這些字對「查電影資料庫」只會幫倒忙。而且如果不清理，
同一部電影會因為場次不同而在清單裡出現五、六次。

寫三條規則反覆套用，直到片名不再變化：

In [ ]:
import re

_PAREN = re.compile(r"[（(][^（()）]*[)）]")                        # (國語版)、(1995)
_TAIL = re.compile(r"[\s　_]+\S*(場|版|重映|加映|上映)\d*\s*$")     # 「…場」「…版」結尾
_FORMAT = re.compile(r"[\s　]+(4K|IMAX|3D|2D|DTS|ATMOS)\s*$", re.I)  # 放映格式


def clean_title(name):
    """把影城片名整理成適合查詢電影資料庫的形式。"""
    title = str(name or "").strip()
    previous = None
    while previous != title:          # 反覆剝，因為註記常常疊在一起
        previous = title
        title = _PAREN.sub(" ", title).strip()
        title = _TAIL.sub("", title).strip()
        title = _FORMAT.sub("", title).strip()
        title = re.sub(r"\s{2,}", " ", title)
    return title


for sample in ["驀然回首 友誼場", "藍色監獄 Ado應援特別場",
               "劇場版 吉伊卡哇 人魚島的秘密 (國語版)",
               "攻殼機動隊(1995) 4K 數位修復版 粉絲紀念場",
               "蜘蛛人：重生日"]:
    print(f"  {pad(sample, 40)} -> {clean_title(sample)!r}")

In [ ]:
if raw:
    cleaned = list(dict.fromkeys([clean_title(n) for n in names if clean_title(n)]))
    print(f"清理前 {len(names)} 筆 -> 清理並去重後 {len(cleaned)} 部")
    print()
    for i, title in enumerate(cleaned[:15], 1):
        print(f"  {i:2}. {title}")
    print("   ...")

**先清理再去重，順序不能反。** 同一部片的六個場次要先被整理成同一個字串，
`dict.fromkeys()` 才有辦法把它們併成一筆。

順帶一提，`nameAlternative` 那邊也有髒資料：`'Blue Lock '` 和 `'Blue Lock'`
（一個結尾多了空白）在原始資料裡是分開的兩筆。這種只差一個空白的假重複，
不 `strip()` 就會變成清單裡的兩部電影。

---
## 5 ・ 統一的網路出入口

到目前為止我們直接呼叫 `requests.get()`。接下來換成 00 寫好的
`movieapp/http.py`，它把所有對外請求收攏到 `fetch_json()` 這一個函式。

好處是**錯誤處理只要寫一次**：逾時、斷線、429、回傳不是 JSON ——
這四種狀況每一支外部 API 都會遇到，收攏之後只有一個地方要處理。

**回傳的是 `(data, error)` 而不是丟例外。**
因為外部 API 失敗是「正常會發生的事」，不是程式寫錯。
把錯誤當成一般的值傳，呼叫端就不必到處包 `try/except`。

In [ ]:
from movieapp import http

data, error = http.fetch_json(
    SHOWTIMES_URL,
    headers=http.BROWSER_HEADERS,        # 瀏覽器偽裝的 header 已經包好了
)

if error:
    print("失敗：", error)
else:
    print("成功，最外層 key:", list(data))
    print("電影數量:", len(find_programs(data)))

---
## 6 ・ 寫成模組

前面的程式碼都還躺在 cell 裡。問題是：**cell 裡的東西沒辦法被別的
notebook 或網頁服務使用。** 第 3 章要用這些片名、第 5 章要接成 API 服務，
如果邏輯只存在 cell 裡，那兩章就只能整段複製 —— 然後就有三份會各自走鐘的程式碼。

`%%writefile` 會把整格的內容存成檔案。執行下面這格，
`movieapp/sources.py` 就誕生了。

> **方向是單向的**：以後要改，請**改這一格再重跑**，
> 不要直接編輯 `movieapp/sources.py` —— 下次重跑這格會蓋掉你的修改。

In [ ]:
%%writefile ../movieapp/sources.py
"""兩家影城的片名抓取。

秀泰   GET   https://capi.showtimes.com.tw/4/app/bootstrap        -> programs[].name
美麗華 POST  https://www.miramarcinemas.tw/api/Booking/GetMovie/  -> TitleAlt

兩支都沒有公開文件，結構是逆向觀察出來的。

本檔案由 notebooks/01_影城API.ipynb 的 %%writefile 產生。
要修改請回去改那一格，不要直接編輯這裡。
"""

import re

from movieapp.http import BROWSER_HEADERS, fetch_json

SHOWTIMES_URL = "https://capi.showtimes.com.tw/4/app/bootstrap"
MIRAMAR_URL = "https://www.miramarcinemas.tw/api/Booking/GetMovie/"

# 影城片名常帶「放映場次」的註記，這些字對查電影資料庫只會幫倒忙：
#   驀然回首 友誼場 / 藍色監獄 Ado應援特別場 / 劇場版 吉伊卡哇 (國語版)
#   攻殼機動隊(1995) 4K 數位修復版 / 超時空甩尾 經典重映
_PAREN = re.compile(r"[（(][^（()）]*[)）]")
_TAIL = re.compile(r"[\s　_]+\S*(場|版|重映|加映|上映)\d*\s*$")
_FORMAT = re.compile(r"[\s　]+(4K|IMAX|3D|2D|DTS|ATMOS)\s*$", re.I)


def collect_keys(obj, target_key, result=None):
    """遞迴走訪整個 JSON，收集所有名為 target_key 的欄位值。"""
    if result is None:
        result = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            if key == target_key:
                result.append(value)
            collect_keys(value, target_key, result)
    elif isinstance(obj, list):
        for item in obj:
            collect_keys(item, target_key, result)

    return result


def clean_title(name):
    """把影城的片名整理成適合查詢電影資料庫的形式。

    反覆套用三條規則直到不再變化，因為註記常常疊在一起
    （例如「攻殼機動隊(1995) 4K 數位修復版 粉絲紀念場」要剝三層）。
    """
    title = str(name or "").strip()
    previous = None
    while previous != title:
        previous = title
        title = _PAREN.sub(" ", title).strip()
        title = _TAIL.sub("", title).strip()
        title = _FORMAT.sub("", title).strip()
        title = re.sub(r"\s{2,}", " ", title)
    return title


def clean_titles(titles):
    """整理片名並去重（保留原始順序）。

    先整理再去重，順序不能反：同一部片的六個場次要先被整理成
    同一個字串，去重才有辦法把它們併成一筆。
    """
    cleaned = [clean_title(t) for t in titles]
    return list(dict.fromkeys([t for t in cleaned if t]))


def find_programs(data):
    """在回應裡找出電影清單。

    為什麼不直接遞迴撈 "name"？因為 name 是個到處都有的通用欄位 ——
    影城名稱、常見問題、場館、類型都叫 name，整份回應裡有 224 個，
    真正是片名的只有一小部分。

    所以先找出「看起來像電影清單」的那個陣列，再從裡面取片名。
    判斷依據是結構而不是路徑：一個 list、元素是 dict、而且帶有
    nameAlternative 欄位 —— 這是電影項目才有的特徵。
    這樣對方就算把 programs 搬到別的層級，程式還是找得到。
    """
    for candidate in collect_keys(data, "programs"):
        if (
            isinstance(candidate, list)
            and candidate
            and isinstance(candidate[0], dict)
            and "nameAlternative" in candidate[0]
        ):
            return candidate
    return []


def showtimes():
    """秀泰影城的片名清單。回傳 (titles, error)。

    用 name（中文片名）而不是 nameAlternative（英文片名）——
    下一章會實測，查電影資料庫時中文名的命中率高很多。
    """
    data, error = fetch_json(SHOWTIMES_URL, headers=BROWSER_HEADERS)
    if error:
        return [], error
    programs = find_programs(data)
    return clean_titles([p.get("name") for p in programs]), None


def miramar():
    """美麗華影城的片名清單。回傳 (titles, error)。

    這支要用 POST（雖然不必帶 body），而且少了 Referer 會被擋。
    """
    headers = dict(BROWSER_HEADERS, Referer="https://www.miramarcinemas.tw/")
    data, error = fetch_json(MIRAMAR_URL, method="POST", headers=headers)
    if error:
        return [], error
    return clean_titles(collect_keys(data, "TitleAlt")), None


# 之後要加第三家影城，只要寫一個同樣形狀的函式再登記到這裡
SOURCES = {
    "showtimes": ("秀泰影城", showtimes),
    "miramar": ("美麗華影城", miramar),
}


def titles_by_source():
    """每家影城各自的片名清單。回傳 ({來源代號: titles}, errors)。

    保留「哪部片在哪家上映」這個資訊，第 3 章合併時要用。
    """
    result = {}
    errors = {}
    for key, (label, fetch) in SOURCES.items():
        titles, error = fetch()
        result[key] = titles
        if error:
            errors[label] = error
    return result, errors


def all_titles():
    """抓所有影城並合併去重。回傳 (titles, errors)。

    某一家掛掉不影響另一家 —— 錯誤收集在 errors 裡回報，
    能拿到的資料照樣回傳。
    """
    by_source, errors = titles_by_source()
    merged = []
    for titles in by_source.values():
        merged.extend(titles)
    return clean_titles(merged), errors

檔案寫好了，import 來用。因為開頭設了 `%autoreload 2`，
以後重跑上面那格，這裡不用重開 kernel 就會拿到新版本。

In [ ]:
from movieapp import sources

titles, error = sources.showtimes()
print(f"秀泰影城：{len(titles)} 部" + (f"（錯誤：{error}）" if error else ""))
for i, title in enumerate(titles[:8], 1):
    print(f"  {i:2}. {title}")
print("  ...")

---
## 7 ・ 美麗華：同樣的套路，不同的坑

美麗華那支有兩個地方跟秀泰不一樣：

1. **要用 POST**（雖然不需要帶任何 body，這種設計在 .NET 後端很常見）
2. **要帶 `Referer`**，否則會被擋

這兩點都不是猜得到的，是試出來的。面對沒有文件的 API，
「用瀏覽器打開網站 → 開 DevTools 的 Network 分頁 → 看它實際送了什麼」
是標準的偵察手法。

好消息是它的 `TitleAlt` 本來就是中文片名，不用像秀泰那樣挑欄位。

In [ ]:
titles, error = sources.miramar()
print(f"美麗華影城：{len(titles)} 部" + (f"（錯誤：{error}）" if error else ""))
for i, title in enumerate(titles[:8], 1):
    print(f"  {i:2}. {title}")
print("  ...")

---
## 8 ・ 合併兩家

`all_titles()` 把兩家串起來。注意它的錯誤處理：
**一家掛掉不會拖垮另一家** —— 失敗的記在 `errors` 裡回報，能拿到的照常回傳。

這在整合多個外部來源時是基本要求，不然任何一家出問題就整個系統掛掉。

In [ ]:
titles, errors = sources.all_titles()

print(f"合併後共 {len(titles)} 部電影")
if errors:
    for label, message in errors.items():
        print(f"  來源錯誤 - {label}：{message}")
else:
    print("兩家來源都正常")

by_source, _ = sources.titles_by_source()
for key, items in by_source.items():
    print(f"  {sources.SOURCES[key][0]}：{len(items)} 部")

兩家加起來會少於各自數量的總和，因為同名的片被去重了。

但**只靠片名字串去重是不夠的** —— 同一部電影在兩家可能寫得不完全一樣。
第 3 章拿到 TMDB 的電影 id 之後，才能精準判斷「這兩筆是不是同一部片」。

---
## 9 ・ 不要依賴外部 API 的回傳順序

同一支 API 連續呼叫兩次，拿到的**內容一樣，順序卻可能不同**。

這不是 bug，是很多 API 的常態（後端可能是多台機器、可能有快取分片）。
下面這格實測一次：

In [ ]:
titles_again, _ = sources.all_titles()

print(f"第一次 {len(titles)} 部｜第二次 {len(titles_again)} 部")
print("內容完全相同？", set(titles) == set(titles_again))
print("連順序都相同？    ", titles == titles_again)

最後一行有可能印出 `False`。**這不是 bug，是一個重要觀念。**

結論：**不要依賴外部 API 的回傳順序**。需要固定順序就自己排序，
這在第 3 章整理資料時會實際處理。比對兩份資料是否一致要用 `set()`，
不是直接比 list。

---
## 小結

- 面對沒有文件的 API，先看原始回應、再決定怎麼解析
- **遞迴找 key 耐撞，但會撈到你沒預期的東西** —— `name` 撈到 224 筆就是教訓，
  要先用結構特徵縮小範圍
- 真實資料很髒：場次後綴、語言版本、畫質標記都要清掉，而且**先清理再去重**
- 不要依賴外部 API 的回傳順序
- 一個來源掛掉不該拖垮其他來源
- 所有網路請求走**同一個出入口**，錯誤處理才不會散落各處
- 用 `%%writefile` 把邏輯變成真的模組，後面的章節和服務才能共用

### 下一章

**02_TMDB** —— 現在只有片名字串，沒有海報、評分、類型、簡介。
下一章拿這些片名去 TMDB 換完整資料，並且會看到一個很重要的陷阱：
**搜尋結果的第一筆，常常不是你要的那部電影。**